1. Import thư viện

In [1]:
import pandas as pd
import numpy as np
import unicodedata

2. Đọc dữ liệu gốc

In [ ]:
df = pd.read_csv('../data/raw/thuviennhadat_raw.csv', dtype=str)
print("Số dòng ban đầu:", len(df))
df.head()

Số dòng ban đầu: 3000


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,"Bán Nhà riêng tạiNgõ Thống Nhất, Phường Trươn...","7,15 tỷ","Ngõ Thống Nhất, Phường Trương Định, Quận Hai B...",30 m²,3,3,6,Sổ đỏ / Sổ hồng,28/09/202512:02
1,"Bán Nhà mặt phố tạiNguyễn Ngọc Nại, Phường Kh...",58 tỷ,"Nguyễn Ngọc Nại, Phường Khương Trung, Quận Tha...",105 m²,NaN,NaN,9,Sổ đỏ / Sổ hồng,28/09/202520:48
2,"Bán Nhà mặt phố tạiNGUYỄN NGỌC NẠI, Phường Kh...",58 tỷ,"NGUYỄN NGỌC NẠI, Phường Khương Trung, Quận Tha...",105 m²,NaN,NaN,9,Sổ đỏ / Sổ hồng,28/09/202520:48
3,"Bán Nhà riêng tạiPhố Vọng, Phường Bạch Mai, Q...","4,85 tỷ","Phố Vọng, Phường Bạch Mai, Quận Hai Bà Trưng, ...",30 m²,2,3,4,NaN,28/09/202510:36
4,"Bán Nhà riêng tạiThanh Lân, Phường Thanh Trì,...","9,10 tỷ","Thanh Lân, Phường Thanh Trì, Quận Hoàng Mai, H...",50 m²,5,NaN,5,Sổ đỏ / Sổ hồng,28/09/202511:49


3. Tiền xử lý và lưu vào file csv
*   **`tieu_de`**: Giữ nguyên kiểu dữ liệu `object` (string), chỉ thực hiện làm sạch bằng cách loại bỏ các ký tự xuống dòng và khoảng trắng thừa.

*   **`gia`**: Chuyển từ kiểu dữ liệu `object` (string) sang `float64` (số thực). Ví dụ chuỗi "2,5 tỷ" sẽ chuyển thành 2.5.

*   **`dia_chi`**: Giữ nguyên kiểu dữ liệu `object` (string).

*   **`dien_tich_dat`**: Chuyển từ kiểu dữ liệu `object` (string) sang `float64` (số thực). Ví dụ chuỗi "125,5 m²" sẽ chuyển thành 125.5.

*   **`phong_ngu`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "3 PN" chuyển thành 3.

*   **`phong_tam`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "3 WC" chuyển thành 3.

*   **`so_tang`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "2 tầng" chuyển thành 2.

*   **`phap_ly`**: Giữ nguyên kiểu dữ liệu `object` (string) nhưng chuỗi chuẩn hóa thành một trong ba loại: "Sổ hồng", "Sổ đỏ", hoặc "Giấy tờ không xác định".
ví dụ nếu có cả sổ hồng và sổ đỏ thì thành sổ hồng, chỉ có sổ hồng => sổ hồng, chỉ có sổ đỏ thì => sổ đỏ, còn các trường hợp còn lại là giấy tờ không xác định.

*   **`ngay_dang`**: Đầu tiên là loại bỏ phần giờ phút giây ở phía sau chỉ giữ lại ngày tháng năm. Tiếp theo chuyển từ kiểu dữ liệu `object` (string) sang `datetime64[ns]`. Ví dụ chuỗi "dd/mm/yyyy" thành kiểu dữ liệu datetime (yyyy-mm-dd).

*   Loại bỏ dòng thiếu dữ liệu quan trọng

*   Xuất file CSV đã làm sạch

In [ ]:
#1. tieu_de
df['tieu_de'] = df['tieu_de'].str.strip().str.replace(r'\s+', ' ', regex=True)

#2. gia → float (tỷ)
def convert_gia(x):
    if pd.isna(x):
        return np.nan
    text = str(x).replace(',', '.').replace(' tỷ', '').strip()
    try:
        return float(text)
    except:
        return np.nan

df['gia'] = df['gia'].apply(convert_gia)

#3. dia_chi
df['dia_chi'] = df['dia_chi'].astype(str)

#4. dien_tich_dat → float
df['dien_tich_dat'] = df['dien_tich_dat'].str.replace(' m²', '', regex=False) \
                                         .str.replace(',', '.', regex=False) \
                                         .str.strip() \
                                         .astype(float, errors='ignore')
df['dien_tich_dat'] = pd.to_numeric(df['dien_tich_dat'], errors='coerce')

#5. phong_ngu → float
df['phong_ngu'] = df['phong_ngu'].str.extract(r'(\d+)')

#6. phong_tam → float 
df['phong_tam'] = df['phong_tam'].str.replace(' WC', '', regex=False) \
                                 .str.extract(r'(\d+)')

#7. so_tang → float
df['so_tang'] = df['so_tang'].str.extract(r'(\d+)')

#8. phap_ly → chuẩn hóa chính xác
def clean_phap_ly(s):
    if pd.isna(s) or str(s).strip() == "":
        return "Giấy tờ không xác định"
    
    # Chuẩn hóa Unicode, loại dấu
    s = str(s).strip()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    
    # So khớp các loại pháp lý
    if "so do" in s:
        return "Sổ đỏ"
    elif "so hong" in s:
        return "Sổ hồng"
    else:
        return "Giấy tờ không xác định"

# Áp dụng trực tiếp trên cột phap_ly
df["phap_ly"] = df["phap_ly"].apply(clean_phap_ly)
#9. ngay_dang → datetime (xử lý cả trường hợp dính giờ)
def parse_ngay_dang(text):
    if pd.isna(text):
        return pd.NaT
    s = str(text).strip()
    # Trường hợp có giờ dính liền kiểu: 27/11/202506:42 → tách lấy 10 ký tự đầu
    if len(s) >= 10 and s[2] == '/' and s[5] == '/':
        date_str = s[:10]  # Lấy đúng dd/mm/yyyy
        try:
            return pd.to_datetime(date_str, format='%d/%m/%Y', dayfirst=True)
        except:
            return pd.NaT
    else:
        try:
            return pd.to_datetime(s, dayfirst=True)
        except:
            return pd.NaT

df['ngay_dang'] = df['ngay_dang'].apply(parse_ngay_dang)
df['ngay_dang'] = df['ngay_dang'].dt.strftime('%Y-%m-%d')

#Ép kiểu cuối cùng 
df = df.astype({
    'tieu_de': 'string',
    'dia_chi': 'string',
    'phap_ly': 'string',
    'ngay_dang': 'datetime64[ns]'
})

# In thử kết quả để kiểm tra
print(df[['phap_ly', 'ngay_dang']].head(10))
print("\nKiểu dữ liệu:")
print(df.dtypes)

df = df.dropna(subset=['phong_ngu', 'phong_tam', 'so_tang','gia'])
df = df.reset_index(drop=True)

# Lưu lại file đã làm sạch (nếu cần)
df.to_csv('../data/interim/thuviennhadat_interim.csv', index=False, encoding='utf-8-sig')

                  phap_ly  ngay_dang
0                 Sổ hồng 2025-09-28
1                 Sổ hồng 2025-09-28
2                 Sổ hồng 2025-09-28
3  Giấy tờ không xác định 2025-09-28
4                 Sổ hồng 2025-09-28
5                 Sổ hồng 2025-09-28
6                 Sổ hồng 2025-09-28
7                 Sổ hồng 2025-09-28
8                 Sổ hồng 2025-09-28
9                 Sổ hồng 2025-12-03

Kiểu dữ liệu:
tieu_de          string[python]
gia                     float64
dia_chi          string[python]
dien_tich_dat           float64
phong_ngu                object
phong_tam                object
so_tang                  object
phap_ly          string[python]
ngay_dang        datetime64[ns]
dtype: object
